# Where long problems break: a probe diagnosis of SIM-CoT GPT-2

This is evaluation only. Every criterion is frozen in `results/step_probe/PLAN.md`, and the notebook refuses to start
unless it matches `PLAN.sha256`.

1. **Validation:** greedy on the 500 validation questions, keeping hidden states at `<bot>`, the 6 latent steps and
   `<eot>`. The answers must match step selection's Stage 0 greedy answers on all 500.
2. **Training, at most 12 minutes:** the same on up to 1,500 GSM8K-Aug training questions for each step count from 1 to 6.
3. **Probes:** linear probes read each intermediate result and the answer from those states, cross-fitted.
4. **Verdict:** on wrong questions, do the final latents hold the gold answer (**READOUT-LIMITED**) or the model's own
   wrong answer (**COMPUTE-LIMITED**)? Where does the chain first stop being readable (**EARLY**, **LATE** or
   **SPREAD**)?

**Before running:** Runtime → Change runtime type → **G4**, then Runtime → **Run all**. If Colab disconnects, Run all again
resumes; one resume is allowed.

In [ ]:
REPO_REF = "main"         # the branch or commit holding scripts/step_probe and the frozen plan

In [ ]:
from pathlib import Path
import hashlib, json, os, runpy, shutil, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/dynamic-state-anchoring")
BACKUP_DIR = DRIVE / "step-probe"
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode:
    raise SystemExit("No GPU: Runtime > Change runtime type > GPU, then Runtime > Run all.")
print("GPU:", gpu.stdout.strip())

PINNED = BACKUP_DIR / "commit.txt"  # a started run keeps the code it started with
ref = PINNED.read_text().strip() if PINNED.exists() else REPO_REF
REPO_DIR = Path("/content/dynamic-state-anchoring")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/sttruji/dynamic-state-anchoring.git", str(REPO_DIR)],
                   check=True)
subprocess.run(["git", "fetch", "origin", ref], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
plan = Path("results/step_probe/PLAN.md")
pinned = Path("results/step_probe/PLAN.sha256").read_text().split()[0]
if hashlib.sha256(plan.read_bytes()).hexdigest() != pinned:
    raise SystemExit("PLAN.md does not match PLAN.sha256: the frozen plan was changed; stop.")
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
PINNED.write_text(COMMIT)
sessions = BACKUP_DIR / "sessions.txt"
count = int(sessions.read_text()) + 1 if sessions.exists() else 1
if count > 2:
    raise SystemExit("The diagnosis allows one session and one resume; this would be a third.")
sessions.write_text(str(count))
print("Code:", COMMIT, f"(from {ref}); plan sha256 matches; session {count} of at most 2")

In [ ]:
VENV = Path("/content/dsa-venv")
subprocess.run([sys.executable, "scripts/setup_colab.py", "--venv-dir", str(VENV),
                "--requirements", "requirements-colab.txt"], check=True)
PYTHON = str(VENV / "bin/python")
os.environ["DSA_DATA_DIR"] = str(DRIVE / "data")
os.environ["HF_HOME"] = "/content/hf-cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
run_logged = runpy.run_path("scripts/colab_process.py")["run_logged"]

def run(*args):
    run_logged([PYTHON, *map(str, args)], cwd=REPO_DIR, log_dir=BACKUP_DIR / "logs")

TRAIN = DRIVE / "data" / "gsm_train.json"  # the GSM8K-Aug training file the inner-loop runs used
assert TRAIN.exists(), f"{TRAIN} is missing; it is the training file from the inner-loop notebook's runs"
VALID = DRIVE / "data" / "bench" / "eval" / "gsm8k_valid.jsonl"  # the locked validation file step selection used
assert VALID.exists(), f"{VALID} is missing; run scripts/setup_data.py as the step selection notebook did"
CHECKPOINT = DRIVE / "checkpoints/simcot-gpt2"
run("scripts/latent_gate/fetch_simcot.py", "--output", CHECKPOINT)

In [ ]:
RUN = BACKUP_DIR / "run"
run("scripts/step_probe/run.py", "--checkpoint", CHECKPOINT, "--output", RUN, "--train-file", TRAIN,
    "--reference", "results/step_select/stage0/evals/seed1.jsonl", "--device", "cuda")
print(json.dumps(json.loads((RUN / "status.json").read_text()), indent=1))

In [ ]:
report = subprocess.run([PYTHON, "scripts/step_probe/report.py", str(RUN)], cwd=REPO_DIR,
                        capture_output=True, text=True)
print(report.stdout, report.stderr)
(BACKUP_DIR / "report.txt").write_text(report.stdout)

In [ ]:
# The records, readings, status and report, for the repo's results folder. The hidden states (about 0.5 GB) stay on
# Drive at step-probe/run.
from google.colab import files

bundle = Path("/content/step-probe")
shutil.rmtree(bundle, ignore_errors=True)
shutil.copytree(RUN, bundle / "run", ignore=shutil.ignore_patterns("*.npy"))
for name in ("report.txt", "commit.txt", "sessions.txt"):
    shutil.copy2(BACKUP_DIR / name, bundle / name)
files.download(shutil.make_archive(str(bundle), "zip", bundle))
print(f"Downloading {bundle}.zip. If it does not start, use the Files panel on the left.")

**Reading the result**
- **Identity** must be 500 of 500.
- **Sensitivity s** is how often the answer probe reads gold on questions the model gets right. Under 0.5, the verdict is
  INCONCLUSIVE.
- **g and w** are, on wrong questions, how often the final latents read as gold and as the model's own answer.
- **The last line** is the validation verdict and break profile.

**Sharing:** the last cell downloads a zip. Point Claude at it.